<a href="https://colab.research.google.com/github/rakshitshah280701/InstructAware/blob/main/InstructAware_MetricColab_SbertCosine_BertScore_Comet.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Create Metrics (SBert,BERTScore, COMET ) on Testing Results

Absolutely! Here's an expanded version of your new description, followed by detailed explanations of each evaluation metric (SBERT Cosine Similarity, BERTScore, and COMET), just like in the previous writeup:

---

### Description of Evaluation Script

This evaluation script is designed to assess the **quality of generated narratives** by comparing **predicted narratives** to **original reference narratives** using **advanced semantic similarity and machine-learned evaluation metrics**. The input to the script is a **CSV file** that includes:

- **Image identifiers or visual data references**
- **Associated vision data**
- **Original (reference) narrative**
- **Predicted (model-generated) narrative**

For each sample in the dataset, the script computes three evaluation metrics:

- **SBERT Cosine Similarity**
- **BERTScore**
- **COMET**

It then appends the computed results as **new columns** to the original dataset and exports the augmented data as a new CSV file. This enriched output allows for comprehensive qualitative and quantitative analysis of how well the predicted narratives align with the original ones.

> Note: Even though three metrics are computed, only **two** columns may be added if some scores are composite (e.g., BERTScore’s F1) or aggregated.

---

## Evaluation Metrics

### 1. **SBERT Cosine Similarity (SbertCosine)**

**SBERT** (Sentence-BERT) is a modification of the BERT architecture that generates **fixed-size sentence embeddings**, allowing fast and effective semantic comparisons between sentences. This script uses **cosine similarity** to compare the SBERT embeddings of the original and predicted narratives.

- **Why use SBERT Cosine?**  
  SBERT is optimized for **semantic textual similarity tasks** and provides highly efficient sentence-level embeddings that reflect contextual meaning. Cosine similarity is used to measure the closeness of these embeddings.
- **Output**: A score between -1 and 1 (but usually between 0 and 1 in practice), where higher values indicate greater semantic similarity.

---

### 2. **BERTScore**

**BERTScore** is an evaluation metric that aligns tokens in the candidate and reference texts using **contextualized embeddings** from a pretrained BERT model. It computes **precision**, **recall**, and **F1** scores based on the similarity of each token pair.

- **Why use BERTScore?**  
  Unlike BLEU or METEOR, BERTScore compares word embeddings rather than word forms. This allows it to handle **synonyms**, **paraphrasing**, and **contextual meaning**, making it more robust to variation in phrasing.
- **Output**: The script usually selects the **F1 score** (between 0 and 1) as the final value for evaluation, indicating the harmonic mean of precision and recall.

---

### 3. **COMET**

**COMET** (Crosslingual Optimized Metric for Evaluation of Translation) is a **learned evaluation model** trained on **human judgment data**. It combines contextual embeddings and attention mechanisms to estimate how close a predicted sentence is to a reference, optionally taking a source input (which could be vision context or a prompt).

- **Why use COMET?**  
  COMET is among the most **human-aligned** metrics available. It goes beyond surface form and embeds both **semantic understanding** and **grammaticality**, reflecting human-like assessments of fluency and adequacy.
- **Output**: A continuous score (typically between 0 and 1) that correlates with human judgment. Higher values indicate stronger narrative quality and alignment.

---

## 📊 Comparison of Evaluation Metrics

| Metric       | Type                    | Captures Semantics? | Handles Synonyms? | Context-Aware | Output Range | Best For |
|--------------|-------------------------|----------------------|--------------------|----------------|--------------|----------|
| **SBERT Cosine** | Sentence Embedding       | ✅ Yes               | ✅ Yes             | ✅ Yes         | 0 to 1       | Fast, accurate semantic similarity |
| **BERTScore**    | Token Embedding Comparison | ✅ Yes               | ✅ Yes             | ✅ Yes         | 0 to 1 (F1)  | Evaluating token-level match with context |
| **COMET**        | Learned Metric (neural)   | ✅ Yes               | ✅ Yes             | ✅ Yes         | 0 to 1       | Human-like assessment of fluency, meaning, and quality |






In [ ]:
!pip install --upgrade --force-reinstall numpy
!pip install --upgrade --force-reinstall torch torchvision torchaudio
!pip uninstall -y unbabel-comet sentence-transformers comet
!pip install unbabel-comet sentence-transformers comet
!pip install --upgrade bert-score


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!pip install bert-score # Re-install bert_score to ensure it's available

In [ ]:
!pip install --upgrade --force-reinstall pandas

In [ ]:
import pandas as pd
import torch
from sentence_transformers import SentenceTransformer, util
from bert_score import score as bert_score
from comet import download_model, load_from_checkpoint

# Load CSV file
# file_path = "/content/drive/MyDrive/InstructAware/Code/Option2TransformerT5_withFixed_TrainTestSplit/Predicted_Narratives_T5model.csv"
file_path = "/content/drive/MyDrive/InstructAware/Code/Option4TransformerDeepSeek/7thAprilRun/New_Output_DeepseekModel.csv"

df = pd.read_csv(file_path)

# Ensure required columns exist
if "ORIGINAL OUTPUT TEXT" not in df.columns or "PREDICTED OUTPUT TEXT" not in df.columns:
    raise ValueError("Missing required columns in CSV file. Ensure 'Original output' and 'Predicted text' are present.")

# Load SBERT model for Cosine Similarity
sbert_model = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')

# Compute SBERT+Cosine Similarity
def compute_sbert_cosine(original, predicted):
    emb1 = sbert_model.encode(original, convert_to_tensor=True)
    emb2 = sbert_model.encode(predicted, convert_to_tensor=True)
    return float(util.pytorch_cos_sim(emb1, emb2).item())

df["SBERT_Cosine"] = df.apply(lambda row: compute_sbert_cosine(str(row["ORIGINAL OUTPUT TEXT"]), str(row["PREDICTED OUTPUT TEXT"])), axis=1)

# Compute BERTScore
df["ORIGINAL OUTPUT TEXT"] = df["ORIGINAL OUTPUT TEXT"].fillna("").astype(str)
df["PREDICTED OUTPUT TEXT"] = df["PREDICTED OUTPUT TEXT"].fillna("").astype(str)
P, R, F1 = bert_score(df["PREDICTED OUTPUT TEXT"].tolist(), df["ORIGINAL OUTPUT TEXT"].tolist(), lang="en", model_type="bert-base-uncased")
df["BERTScore_F1"] = F1.tolist()

### **Compute COMET Scores**
# Download and load the COMET model
comet_model_path = download_model("Unbabel/wmt22-comet-da")  # Pre-trained model for evaluation
comet_model = load_from_checkpoint(comet_model_path)

# Prepare input data for COMET
comet_inputs = [
    {"src": "", "mt": pred, "ref": ref}
    for pred, ref in zip(df["PREDICTED OUTPUT TEXT"], df["ORIGINAL OUTPUT TEXT"])
]

# Compute COMET scores
comet_scores = comet_model.predict(comet_inputs, batch_size=8)
df["comet_score"] = comet_scores["scores"]


# Save results
# output_file = "/content/drive/MyDrive/InstructAware/Code/Option2TransformerT5_withFixed_TrainTestSplit/Predicted_Narratives_T5model_Metric_SBERTCOSINE_BERTSCORE_COMET.csv"
output_file = "/content/drive/MyDrive/InstructAware/Code/Option4TransformerDeepSeek/7thAprilRun/New_Output_DeepseekModel_sbertCosine_BertScore_Comet.csv"
df.to_csv(output_file, index=False)



In [ ]:
import pandas as pd

# Load the CSV file
csv_path = "/content/drive/MyDrive/InstructAware/Code/Option4TransformerDeepSeek/7thAprilRun/New_Output_DeepseekModel_sbertCosine_BertScore_Comet.csv"  # Update the path if needed
df = pd.read_csv(csv_path)

# Display first few rows
df.head()

from google.colab import data_table

# Display CSV as an interactive table
data_table.DataTable(df)